# Overwatch RL Lab: learning curves from real game logs

Every number here comes from log files that Overwatch wrote while the Workshop mode ran
(`data/logs/*.txt`). Nothing is simulated. To add your own session, copy the newest file from
`Documents\Overwatch\Workshop\` into `data/logs/` and run all cells again.

Column meanings: `docs/logging.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "logs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "analysis"))

import pandas as pd
from rllab.logs import parse_log, episodes_frame, EXPERIMENTS
from rllab.plots import summary_table, learning_curves
from rllab.replay import replay

files = sorted((ROOT / "data" / "logs").glob("*.txt"))
log = parse_log(*files)
episodes = episodes_frame(log)
print(f"{len(files)} log files, {len(log.ep)} episodes, {len(log.step)} logged steps")
assert not log.skipped, log.skipped[:3]

5 log files, 313 episodes, 7100 logged steps


## 1. What was measured about the arena

One `CAL` line per run: the island's edges as found by ray casts, and Tracer's timed run speed.

In [2]:
pd.DataFrame(log.cal)[["file", "x_min", "x_max", "z_min", "z_max", "floor_y", "cell", "run_speed", "coast_dist", "act_duration"]].drop_duplicates()

,file,x_min,x_max,z_min,z_max,floor_y,cell,run_speed,coast_dist,act_duration
0,2026-11-21_blink_fixed_duration_qlearning.txt,-20,20,-20,20,0,2,5.98,0.25,0.33
1,2026-11-21_cliffwalk_qlearning_then_controls.txt,-20,20,-20,20,0,2,5.98,0.25,0.33
4,2026-11-21_cliffwalk_sarsa.txt,-20,20,-20,20,0,2,5.98,0.25,0.33
5,2026-11-21_demo_patrol.txt,-20,20,-20,20,0,2,5.98,0.25,0.33
6,2026-11-21_widowmaker_sarsa.txt,-20,20,-20,20,0,2,5.98,0.25,0.33


## 2. Did the game compute the updates correctly?

Where per-step logging was on, the replay applies the reference update
(`analysis/rllab/reference.py`) to the transitions the game logged and compares every target and
every Q-value with what the game wrote. A mismatch would be listed here.

In [3]:
report = replay(log)
print(f"steps checked: {report.steps_checked}, updates checked: {report.updates_checked}, episodes re-added: {report.episodes_checked}")
print(dict(report.by_algo))
if report.training_steps:
    print(f"exploratory draws: {report.explore_steps / report.training_steps:.1%} of training steps")
print("mismatches:", report.mismatches[:5] if report.mismatches else "none")

steps checked: 6978, updates checked: 6085, episodes re-added: 313
{'Q-learning': 2968, 'SARSA': 3117}
exploratory draws: 10.0% of training steps
mismatches: none


## 3. Episodes per run

One row per file, run, algorithm and mode. `fall_rate` and `goal_rate` are shares of episodes.
`mean_edge_dist` is Tracer's average distance from the island edge in metres (row 0 is centred
1.5 m from the edge, row 1 at 3.5 m).

In [4]:
runs = pd.DataFrame(log.run)
runs["experiment_name"] = runs["experiment"].map(EXPERIMENTS)
learning = episodes.merge(runs[["file", "run", "experiment_name", "closed_loop", "use_blink"]], on=["file", "run"])
summary_table(learning).round(2)

,file,run,algo_name,mode_name,episodes,mean_return,goal_rate,fall_rate,mean_steps,mean_edge_dist,epsilon,alpha,gamma
0,2026-11-21_blink_fixed_duration_qlearning.txt,1,Q-learning,training,29,-18.72,0.59,0.41,6.66,2.05,0.1,0.5,1.0
1,2026-11-21_cliffwalk_qlearning_then_controls.txt,1,Q-learning,training,119,-15.76,0.71,0.28,23.32,3.06,0.1,0.5,1.0
2,2026-11-21_cliffwalk_qlearning_then_controls.txt,1,Q-learning,evaluation,12,37.00,1.00,0.00,13.00,2.39,0.1,0.5,1.0
3,2026-11-21_cliffwalk_qlearning_then_controls.txt,2,SARSA,evaluation,3,-132.00,0.00,1.00,32.00,3.42,0.1,0.5,1.0
4,2026-11-21_cliffwalk_qlearning_then_controls.txt,3,SARSA,evaluation,1,-101.00,0.00,1.00,1.00,0.00,0.1,0.5,1.0
5,2026-11-21_cliffwalk_sarsa.txt,1,SARSA,training,115,-3.17,0.81,0.19,24.47,3.91,0.1,0.5,1.0
6,2026-11-21_cliffwalk_sarsa.txt,1,SARSA,evaluation,3,-150.00,0.00,0.00,100.00,3.43,0.1,0.5,1.0
7,2026-11-21_demo_patrol.txt,1,SARSA,evaluation,20,33.00,1.00,0.00,17.00,6.45,0.1,0.5,1.0
8,2026-11-21_widowmaker_sarsa.txt,1,SARSA,training,11,-127.55,0.00,0.64,27.55,3.92,0.1,0.5,1.0


## 4. Learning curves (training episodes, cliff walk only)

Top: return per episode. Middle: share of episodes that ended in a fall. Bottom: mean distance
from the edge. Each line is one run, smoothed over 20 episodes; the legend gives n. Only runs on
the standard arena are drawn (closed-loop moves, four actions).

One run per algorithm is a single sample, and the Workshop cannot seed its random numbers.
Treat a gap between the lines as a lead to repeat, not as a result.

In [5]:
# Standard arena only: closed-loop moves, four actions. Blink and fixed-duration runs are a different environment.
standard = (learning["closed_loop"] == True) & (learning["use_blink"] == False)
cliff = learning[(learning["experiment_name"] == "cliff walk") & (learning["mode_name"] == "training") & standard]
if len(cliff):
    figure = learning_curves(cliff, window=20)
    figure.savefig(ROOT / "analysis" / "figures" / "learning_curves.png", dpi=160)
    display(figure)
else:
    print("No cliff-walk training episodes in data/logs yet.")

<Figure size 800x900 with 3 Axes>

## 5. SARSA against Q-learning, late training

The last 40 training episodes of each cliff-walk run. This is the comparison Lab 02 asks about:
return, fall rate and distance from the edge while both are still exploring.

In [6]:
late = cliff.sort_values("episode").groupby(["file", "run", "algo_name"]).tail(40)
table = late.groupby(["algo_name", "file", "run"]).agg(
    episodes=("ret", "size"), mean_return=("ret", "mean"), fall_rate=("fell", "mean"),
    goal_rate=("goal", "mean"), mean_steps=("steps", "mean"), mean_edge_dist=("mean_edge_dist", "mean"),
).round(2)
table

,,,episodes,mean_return,fall_rate,goal_rate,mean_steps,mean_edge_dist
algo_name,file,run,,,,,,
Q-learning,2026-11-21_cliffwalk_qlearning_then_controls.txt,1,40,-1.3,0.25,0.75,13.8,2.40
SARSA,2026-11-21_cliffwalk_sarsa.txt,1,40,18.2,0.10,0.90,16.8,3.88


## 6. Greedy evaluation

Episodes run with exploration off and learning frozen (Reload in game). If there are none yet,
this table is empty.

In [7]:
evaluation = learning[learning["mode_name"] == "evaluation"]
evaluation.groupby(["experiment_name", "algo_name", "file", "run"]).agg(
    episodes=("ret", "size"), mean_return=("ret", "mean"), goal_rate=("goal", "mean"),
    fall_rate=("fell", "mean"), mean_steps=("steps", "mean"), mean_edge_dist=("mean_edge_dist", "mean"),
).round(2)

episodes  \
experiment_name algo_name  file                                             run             
arena demo      SARSA      2026-11-21_demo_patrol.txt                       1          20   
cliff walk      Q-learning 2026-11-21_cliffwalk_qlearning_then_controls.txt 1          12   
                SARSA      2026-11-21_cliffwalk_qlearning_then_controls.txt 2           3   
                                                                            3           1   
                           2026-11-21_cliffwalk_sarsa.txt                   1           3   

                                                                                 mean_return  \
experiment_name algo_name  file                                             run                
arena demo      SARSA      2026-11-21_demo_patrol.txt                       1           33.0   
cliff walk      Q-learning 2026-11-21_cliffwalk_qlearning_then_controls.txt 1           37.0   
                SARSA      2026-11-21_cliffwalk_qlearning_then_controls.txt 2         -132.0   
                                                                            3         -101.0   
                           2026-11-21_cliffwalk_sarsa.txt                   1         -150.0   

                                                                                 goal_rate  \
experiment_name algo_name  file                                             run              
arena demo      SARSA      2026-11-21_demo_patrol.txt                       1          1.0   
cliff walk      Q-learning 2026-11-21_cliffwalk_qlearning_then_controls.txt 1          1.0   
                SARSA      2026-11-21_cliffwalk_qlearning_then_controls.txt 2          0.0   
                                                                            3          0.0   
                           2026-11-21_cliffwalk_sarsa.txt                   1          0.0   

                                                                                 fall_rate  \
experiment_name algo_name  file                                             run              
arena demo      SARSA      2026-11-21_demo_patrol.txt                       1          0.0   
cliff walk      Q-learning 2026-11-21_cliffwalk_qlearning_then_controls.txt 1          0.0   
                SARSA      2026-11-21_cliffwalk_qlearning_then_controls.txt 2          1.0   
                                                                            3          1.0   
                           2026-11-21_cliffwalk_sarsa.txt                   1          0.0   

                                                                                 mean_steps  \
experiment_name algo_name  file                                             run               
arena demo      SARSA      2026-11-21_demo_patrol.txt                       1          17.0   
cliff walk      Q-learning 2026-11-21_cliffwalk_qlearning_then_controls.txt 1          13.0   
                SARSA      2026-11-21_cliffwalk_qlearning_then_controls.txt 2          32.0   
                                                                            3           1.0   
                           2026-11-21_cliffwalk_sarsa.txt                   1         100.0   

                                                                                 mean_edge_dist  
experiment_name algo_name  file                                             run                  
arena demo      SARSA      2026-11-21_demo_patrol.txt                       1              6.45  
cliff walk      Q-learning 2026-11-21_cliffwalk_qlearning_then_controls.txt 1              2.39  
                SARSA      2026-11-21_cliffwalk_qlearning_then_controls.txt 2              3.42  
                                                                            3              0.00  
                           2026-11-21_cliffwalk_sarsa.txt                   1              3.43